# 1.2. Análisis, implementación y prueba de un caso práctico con Arduino para comprender el fundamento de internet de las cosas

## Introducción de la actividad.

Es importante conocer cómo funciona el Internet de las Cosas, llevar a cabo un caso práctico ayudará a comprender cómo por medio de programación aplicada en hardware (dispositivos), se pueden manipular objetos. Es importante mencionar que llevar a cabo un caso totalmente práctico, implica adquirir dispositivos, tarjetas, sensores, programar, etc., para ello es importante conocer Arduino y aplicar un caso práctico que solucione un problema orientado a internet de las cosas.

## Propósito de la actividad

Comprender el fundamento de internet de las cosas mediante un caso práctico con Arduino.

## Instrucciones.

Analicé, implemente y evalué un caso práctico con Arduino para entender el funcionamiento de internet de las cosas.

## Datos generales

**Alumno:** Marco Sebastián Hernández Parada

**Caso práctico:** Monitoreo remoto de temperatura y humedad con alerta y control de ventilación mediante ESP32 y MQTT.

**Simulador:** Wokwi (https://wokwi.com)

**Enlace al proyecto:** https://wokwi.com/projects/478346489118390273

## 1. Análisis del caso práctico

### 1.1 Planteamiento del problema

En espacios como cuartos de servidores, invernaderos, bodegas de alimentos o farmacias, la temperatura y la humedad deben mantenerse dentro de un rango. Normalmente se revisan de forma manual, por lo que un aumento de temperatura puede pasar inadvertido hasta que ya causó daños (equipos sobrecalentados, producto echado a perder).

Se propone un dispositivo IoT que mide continuamente estas variables, las envía por internet para consultarlas en tiempo real desde cualquier lugar, activa una alerta local cuando se supera un umbral y permite encender o apagar un ventilador y cambiar el umbral de forma remota.

### 1.2 Objetivos

- Medir temperatura y humedad con un sensor conectado a un microcontrolador.
- Transmitir las mediciones por internet usando un protocolo propio de IoT (MQTT).
- Actuar sobre el entorno de dos formas: automáticamente (alerta y ventilador en modo AUTO) y por orden remota (ventilador ON/OFF y cambio de umbral).
- Probar el funcionamiento completo en un simulador antes de llevarlo a hardware real.

### 1.3 Fundamento: arquitectura de Internet de las Cosas

Un sistema IoT suele describirse en tres capas. La tabla muestra cómo se cubre cada una en este caso.

| Capa | Función | Elemento en el caso práctico |
|---|---|---|
| Percepción (dispositivos) | Medir el entorno y actuar sobre él | Sensor DHT22, LED rojo (alerta), LED azul (ventilador) |
| Red (comunicación) | Transportar los datos entre el dispositivo y la nube | ESP32 con WiFi, protocolo MQTT sobre TCP/IP |
| Aplicación | Mostrar los datos y enviar órdenes al usuario | Broker HiveMQ y cliente web MQTT |

El flujo de información es el siguiente:

1. El DHT22 mide temperatura y humedad.
2. El ESP32 lee el sensor cada 5 segundos, decide si hay alerta y controla los LEDs.
3. El ESP32 publica las mediciones en formato JSON en el broker MQTT a través de WiFi.
4. Cualquier cliente suscrito (navegador, celular, dashboard) recibe los datos en tiempo real.
5. El usuario publica comandos en el broker y el ESP32, que está suscrito, los recibe y actúa.

### 1.4 Protocolo MQTT

MQTT (Message Queuing Telemetry Transport) es un protocolo ligero de tipo publicación/suscripción, diseñado para dispositivos con pocos recursos y redes poco confiables, por lo que es uno de los más usados en IoT. Sus elementos principales son:

- **Broker:** servidor intermediario que recibe los mensajes y los reenvía a los suscriptores. En este caso se usa el broker público `broker.hivemq.com`.
- **Tópico:** "canal" jerárquico con el que se etiquetan los mensajes, por ejemplo `mshp/iot/p12/telemetria`.
- **Publicador y suscriptor:** los clientes no se conocen entre sí; solo publican o se suscriben a tópicos en el broker.
- **QoS, retained y Last Will:** niveles de garantía de entrega, mensajes que el broker guarda para nuevos suscriptores y un mensaje que el broker publica automáticamente si el dispositivo se desconecta sin avisar.

Tópicos definidos para el caso práctico:

| Tópico | Dirección | Contenido |
|---|---|---|
| `mshp/iot/p12/telemetria` | ESP32 hacia la nube | JSON con temperatura, humedad, umbral, alerta, estado y modo del ventilador |
| `mshp/iot/p12/estado` | ESP32 hacia la nube | `online` al conectarse, `offline` (Last Will) si se pierde la conexión |
| `mshp/iot/p12/ventilador/set` | Nube hacia el ESP32 | `ON`, `OFF` o `AUTO` |
| `mshp/iot/p12/umbral/set` | Nube hacia el ESP32 | Número entre -40 y 80 (grados Celsius) |

### 1.5 Componentes

| Componente | Cantidad | Función |
|---|---|---|
| ESP32 DevKit V1 | 1 | Microcontrolador con WiFi integrado, programable con el entorno Arduino |
| Sensor DHT22 | 1 | Mide temperatura (-40 a 80 °C) y humedad relativa (0 a 100 %) |
| LED rojo | 1 | Indicador local de alerta por temperatura alta |
| LED azul | 1 | Simula el ventilador (en un sistema real sería un relé) |
| Resistencias de 220 ohm | 2 | Limitan la corriente de los LEDs |

Se eligió el ESP32 en lugar de un Arduino UNO porque incluye WiFi, lo que permite conectarse a internet sin módulos adicionales, y se programa igual que un Arduino.

## 2. Implementación

### 2.1 Simulador

La implementación se realizó en **Wokwi**, un simulador en línea de Arduino y ESP32 que, a diferencia de otros simuladores, emula la conexión WiFi (red `Wokwi-GUEST`) y permite que el dispositivo se comunique con servicios reales en internet, como un broker MQTT. El proyecto se creó con la plantilla **ESP32 / Arduino**.

### 2.2 Conexiones del circuito

| Componente | Pin del componente | Pin del ESP32 |
|---|---|---|
| DHT22 | VCC | 3V3 |
| DHT22 | SDA (datos) | GPIO 15 (D15) |
| DHT22 | GND | GND |
| LED rojo | Ánodo (a través de resistencia de 220 ohm) | GPIO 2 (D2) |
| LED rojo | Cátodo | GND |
| LED azul | Ánodo (a través de resistencia de 220 ohm) | GPIO 4 (D4) |
| LED azul | Cátodo | GND |

![Circuito en Wokwi](imagenes/Circuito.png)

**Figura 1.** Circuito armado en Wokwi durante la simulación. Con el DHT22 a 32.3 °C (umbral de 30 °C) se encienden el LED rojo (alerta) y el LED azul (ventilador en modo AUTO). En la parte inferior, el monitor serial muestra los mensajes publicados.

### 2.3 Librerías

- `WiFi.h`: incluida en el núcleo del ESP32, maneja la conexión inalámbrica.
- `PubSubClient`: cliente MQTT para Arduino.
- `DHT sensor library for ESPx` (`DHTesp.h`): lectura del sensor DHT22.

### 2.4 Lógica del programa

- **setup():** configura los pines, inicializa el sensor, se conecta al WiFi y prepara el cliente MQTT.
- **loop():** mantiene las conexiones WiFi y MQTT (reconecta si se pierden), atiende los mensajes entrantes y cada 5 segundos lee el sensor, actualiza los LEDs y publica la telemetría.
- **actualizarActuadores():** enciende el LED rojo si la temperatura supera el umbral. El ventilador se enciende o apaga según el modo: `ON` y `OFF` lo fuerzan y `AUTO` lo enciende solo cuando hay alerta.
- **alRecibirMensaje():** procesa los comandos remotos, valida su formato (ignora valores no válidos) y publica de inmediato el nuevo estado.
- **conectarMQTT():** se conecta al broker con un identificador aleatorio y registra el mensaje Last Will `offline`.

### 2.5 Código fuente (sketch.ino)

```cpp
/*
  Practica 1.2 - Monitoreo remoto de temperatura y humedad (IoT)
  Autor: Marco Sebastian Hernandez Parada

  ESP32 + DHT22 + 2 LEDs, comunicacion por WiFi usando MQTT.
  - LED rojo (GPIO 2): alerta local cuando la temperatura supera el umbral.
  - LED azul (GPIO 4): simula un ventilador controlado de forma remota.
*/

#include <WiFi.h>
#include <PubSubClient.h>
#include "DHTesp.h"

// Pines
const int PIN_DHT = 15;
const int PIN_LED_ALERTA = 2;
const int PIN_LED_VENTILADOR = 4;

// Red WiFi simulada de Wokwi
const char* WIFI_SSID = "Wokwi-GUEST";
const char* WIFI_PASS = "";

// Broker MQTT publico
const char* MQTT_BROKER = "broker.hivemq.com";
const int MQTT_PORT = 1883;

// Topicos (el prefijo debe ser unico porque el broker es publico)
#define TOPIC_BASE "mshp/iot/p12"
const char* TOPIC_TELEMETRIA = TOPIC_BASE "/telemetria";
const char* TOPIC_ESTADO = TOPIC_BASE "/estado";
const char* TOPIC_VENTILADOR_SET = TOPIC_BASE "/ventilador/set";
const char* TOPIC_UMBRAL_SET = TOPIC_BASE "/umbral/set";

const unsigned long INTERVALO_MS = 5000;

WiFiClient espClient;
PubSubClient mqtt(espClient);
DHTesp dht;

float umbral = 30.0;
String modoVentilador = "AUTO";  // AUTO, ON u OFF
bool ventiladorEncendido = false;
bool alerta = false;
float ultimaTemp = NAN;
float ultimaHum = NAN;
unsigned long ultimoEnvio = 0;

void conectarWiFi() {
  Serial.print("Conectando a WiFi");
  WiFi.begin(WIFI_SSID, WIFI_PASS, 6);
  while (WiFi.status() != WL_CONNECTED) {
    delay(250);
    Serial.print(".");
  }
  Serial.println();
  Serial.print("WiFi conectado. IP: ");
  Serial.println(WiFi.localIP());
}

void actualizarActuadores() {
  alerta = !isnan(ultimaTemp) && ultimaTemp > umbral;

  if (modoVentilador == "ON") {
    ventiladorEncendido = true;
  } else if (modoVentilador == "OFF") {
    ventiladorEncendido = false;
  } else {
    ventiladorEncendido = alerta;  // AUTO: sigue a la alerta
  }

  digitalWrite(PIN_LED_ALERTA, alerta ? HIGH : LOW);
  digitalWrite(PIN_LED_VENTILADOR, ventiladorEncendido ? HIGH : LOW);
}

void publicarTelemetria() {
  if (isnan(ultimaTemp) || isnan(ultimaHum)) return;

  char payload[200];
  snprintf(payload, sizeof(payload),
           "{\"temperatura\":%.1f,\"humedad\":%.1f,\"umbral\":%.1f,"
           "\"alerta\":%s,\"ventilador\":\"%s\",\"modo\":\"%s\"}",
           ultimaTemp, ultimaHum, umbral,
           alerta ? "true" : "false",
           ventiladorEncendido ? "ON" : "OFF",
           modoVentilador.c_str());

  mqtt.publish(TOPIC_TELEMETRIA, payload);
  Serial.print("Publicado -> ");
  Serial.println(payload);
}

void alRecibirMensaje(char* topic, byte* payload, unsigned int length) {
  String mensaje;
  for (unsigned int i = 0; i < length; i++) mensaje += (char)payload[i];
  mensaje.trim();
  Serial.printf("Mensaje recibido [%s]: %s\n", topic, mensaje.c_str());

  if (strcmp(topic, TOPIC_VENTILADOR_SET) == 0) {
    mensaje.toUpperCase();
    if (mensaje != "ON" && mensaje != "OFF" && mensaje != "AUTO") {
      Serial.println("Comando no valido (use ON, OFF o AUTO)");
      return;
    }
    modoVentilador = mensaje;
  } else if (strcmp(topic, TOPIC_UMBRAL_SET) == 0) {
    char* fin;
    float nuevo = strtof(mensaje.c_str(), &fin);
    if (fin == mensaje.c_str() || *fin != '\0' || nuevo < -40 || nuevo > 80) {
      Serial.println("Umbral no valido (numero entre -40 y 80)");
      return;
    }
    umbral = nuevo;
  }

  actualizarActuadores();
  publicarTelemetria();
}

void conectarMQTT() {
  while (!mqtt.connected()) {
    String clientId = "esp32-mshp-" + String(esp_random(), HEX);
    Serial.print("Conectando a MQTT...");
    // Last Will: si el ESP32 se desconecta, el broker publica "offline"
    if (mqtt.connect(clientId.c_str(), TOPIC_ESTADO, 1, true, "offline")) {
      Serial.println(" conectado");
      mqtt.publish(TOPIC_ESTADO, "online", true);
      mqtt.subscribe(TOPIC_VENTILADOR_SET);
      mqtt.subscribe(TOPIC_UMBRAL_SET);
    } else {
      Serial.printf(" fallo (rc=%d), reintento en 2 s\n", mqtt.state());
      delay(2000);
    }
  }
}

void setup() {
  Serial.begin(115200);
  pinMode(PIN_LED_ALERTA, OUTPUT);
  pinMode(PIN_LED_VENTILADOR, OUTPUT);
  dht.setup(PIN_DHT, DHTesp::DHT22);

  conectarWiFi();
  mqtt.setServer(MQTT_BROKER, MQTT_PORT);
  mqtt.setCallback(alRecibirMensaje);
}

void loop() {
  if (WiFi.status() != WL_CONNECTED) conectarWiFi();
  if (!mqtt.connected()) conectarMQTT();
  mqtt.loop();

  if (millis() - ultimoEnvio >= INTERVALO_MS) {
    ultimoEnvio = millis();
    TempAndHumidity datos = dht.getTempAndHumidity();
    if (dht.getStatus() != DHTesp::ERROR_NONE) {
      Serial.print("Error del sensor: ");
      Serial.println(dht.getStatusString());
      return;
    }
    ultimaTemp = datos.temperature;
    ultimaHum = datos.humidity;
    actualizarActuadores();
    publicarTelemetria();
  }
}
```

## 3. Pruebas

### 3.1 Herramienta de prueba

Para representar la capa de aplicación se usa el cliente web de HiveMQ (https://www.hivemq.com/demos/websocket-client/). Esta página es un cliente MQTT que funciona en el navegador. Permite ver en tiempo real los mensajes que publica el ESP32 y enviarle comandos, como lo haría una aplicación móvil o un dashboard.

### 3.2 Ejecución de la simulación en Wokwi

1. Abrir el proyecto en https://wokwi.com/projects/478346489118390273.
2. Presionar el botón **Play** (Start the simulation).
3. Verificar en el monitor serial que aparezcan, en este orden:
   - `WiFi conectado. IP: ...`: el ESP32 se conectó a la red simulada `Wokwi-GUEST`.
   - `Conectando a MQTT... conectado`: el ESP32 se conectó al broker `broker.hivemq.com`.
   - `Publicado -> {...}`: un mensaje JSON cada 5 segundos con las mediciones.
4. Para variar las condiciones del entorno durante la simulación, dar clic sobre el sensor DHT22 y mover los controles de temperatura y humedad.

### 3.3 Conexión del cliente web al broker

En la sección **Connection** del cliente web de HiveMQ se capturan los siguientes datos y se presiona **Connect**. El indicador de estado cambia a *connected*.

| Campo | Valor | Explicación |
|---|---|---|
| Host | `broker.hivemq.com` | Broker MQTT público, el mismo que usa el ESP32 |
| Port | `8884` | Puerto de MQTT sobre WebSockets seguros |
| ClientID | Valor generado automáticamente | Identificador único del cliente ante el broker |
| Username / Password | Vacíos | El broker público no requiere autenticación |
| Keep Alive | `60` | Segundos máximos sin comunicación antes de considerar perdida la conexión |
| SSL | Activado | Necesario para el puerto 8884 |
| Clean Session | Activado | No conserva suscripciones de sesiones anteriores |
| Last-Will (Topic, QoS, Retain, Message) | Vacíos | No se requieren para el cliente de prueba |

**Diferencia de puertos.** El ESP32 se conecta por el puerto `1883`, que usa MQTT directamente sobre TCP. Un navegador no puede abrir conexiones TCP directas, por lo que el cliente web usa MQTT encapsulado en WebSockets seguros, que en este broker van por el puerto `8884`. Ambos clientes llegan al mismo broker, así que comparten los mismos tópicos y mensajes. Esto muestra una característica de MQTT: dispositivos y aplicaciones distintos se comunican sin conocerse, solo a través del broker.

### 3.4 Suscripción a los datos del dispositivo

1. En la sección **Subscriptions**, presionar **Add New Topic Subscription**.
2. Escribir el tópico `mshp/iot/p12/#`, dejar QoS en 0 y presionar **Subscribe**.
3. En la sección **Messages** aparecen el estado del dispositivo (`online`) y un mensaje de telemetría cada 5 segundos, por ejemplo:

```
mshp/iot/p12/telemetria
{"temperatura":24.0,"humedad":50.0,"umbral":30.0,
 "alerta":false,"ventilador":"OFF","modo":"AUTO"}
```

El carácter `#` es un comodín de varios niveles. Con él, el cliente recibe todos los tópicos bajo `mshp/iot/p12/` (`telemetria`, `estado`, etc.) con una sola suscripción.

### 3.5 Envío de comandos al dispositivo

En la sección **Publish** se captura el tópico y el mensaje y se presiona **Publish**:

| Topic | Message | Efecto en el ESP32 |
|---|---|---|
| `mshp/iot/p12/ventilador/set` | `ON` | Enciende el ventilador (LED azul) sin importar la temperatura |
| `mshp/iot/p12/ventilador/set` | `OFF` | Apaga el ventilador aunque haya alerta |
| `mshp/iot/p12/ventilador/set` | `AUTO` | El ventilador se enciende solo cuando la temperatura supera el umbral |
| `mshp/iot/p12/umbral/set` | `20` (u otro número entre -40 y 80) | Cambia el umbral de alerta en grados Celsius |

Al recibir un comando, el ESP32 lo muestra en el monitor serial (`Mensaje recibido [...]`), actualiza los LEDs y publica de inmediato un nuevo mensaje de telemetría. Así se confirma en el cliente web que la orden se aplicó y se cierra el ciclo de control remoto.

### 3.6 Plan y resultados de pruebas

| No. | Prueba | Acción | Resultado esperado | Resultado obtenido |
|---|---|---|---|---|
| P1 | Conexión WiFi | Iniciar la simulación | El monitor serial muestra "WiFi conectado" y una IP | Aprobada: el ESP32 obtuvo red y publica en el broker (Figuras 1 y 2) |
| P2 | Conexión MQTT | Esperar la conexión al broker | Serial muestra "conectado"; en `estado` llega `online` | Aprobada: el cliente web recibe los mensajes del ESP32 (Figura 2) |
| P3 | Telemetría periódica | Observar el cliente web | Llega un JSON cada 5 s con temperatura 24 y humedad 50 | Aprobada: se publican mensajes JSON consecutivos (Figura 1) y llegan a `telemetria` (Figura 2) |
| P4 | Alerta automática | Subir la temperatura del DHT22 a 35 °C | LED rojo y LED azul encendidos; `"alerta":true` | Aprobada: con 44.5 °C y 32.3 °C ambos LEDs encendidos, `"alerta":true`, `"ventilador":"ON"` (Figuras 1 y 2) |
| P5 | Fin de la alerta | Bajar la temperatura a 25 °C | Ambos LEDs se apagan; `"alerta":false` | Aprobada (verificada en simulación): ambos LEDs se apagan y se publica `"alerta":false` |
| P6 | Control remoto ON | Publicar `ON` en `ventilador/set` | LED azul encendido sin alerta; `"modo":"ON"` | Aprobada (verificada en simulación): LED azul encendido sin alerta, `"modo":"ON"` |
| P7 | Control remoto OFF | Con 35 °C, publicar `OFF` | LED rojo encendido y LED azul apagado | Aprobada (verificada en simulación): LED rojo encendido y LED azul apagado |
| P8 | Cambio de umbral | Con 25 °C, publicar `AUTO` en `ventilador/set` y `20` en `umbral/set` | Se activa la alerta; `"umbral":20.0` | Aprobada (verificada en simulación): se activa la alerta con `"umbral":20.0` |
| P9 | Comando no válido | Publicar `HOLA` en `ventilador/set` | Serial muestra "Comando no valido"; el estado no cambia | Aprobada (verificada en simulación): se muestra "Comando no valido" y el estado no cambia |
| P10 | Desconexión | Detener la simulación | Tras unos segundos llega `offline` en `estado` | Aprobada (verificada en simulación): el broker publica `offline` en `estado` |

### 3.7 Evidencias

La Figura 1 (sección 2.2) muestra el circuito durante la simulación, con los LEDs encendidos por la alerta y el monitor serial publicando la telemetría.

![Cliente web de HiveMQ](imagenes/HiveMQ.png)

**Figura 2.** Cliente web de HiveMQ conectado a `broker.hivemq.com` (estado *connected*) y suscrito a `mshp/iot/p12/#`. En *Messages* se recibe en el tópico `mshp/iot/p12/telemetria` el mismo JSON que publicó el ESP32: temperatura de 32.3 °C, umbral de 30 °C, alerta activa y ventilador encendido en modo AUTO. Esto confirma que el dato viajó del sensor simulado al broker en internet y de ahí a una aplicación externa.

## 4. Evaluación

Se aprobaron las 10 pruebas del plan. P1 a P4 se documentan con las Figuras 1 y 2, y P5 a P10 se verificaron directamente en la simulación. Los resultados comprueban el flujo completo de un sistema IoT:

- **Percepción:** el ESP32 lee correctamente el DHT22. Al modificar el sensor en Wokwi, los nuevos valores (44.5 °C y 32.3 °C) aparecen en la siguiente publicación.
- **Decisión local:** al superar el umbral de 30 °C, el dispositivo activa la alerta y el ventilador por sí mismo, sin esperar una orden de la nube. En modo AUTO, ambos LEDs responden de acuerdo con la lógica programada.
- **Comunicación:** los mensajes JSON publicados por el ESP32 llegan sin alteraciones al cliente web de HiveMQ en el tópico `mshp/iot/p12/telemetria`. El contenido observado en el monitor serial y en el cliente web es idéntico.
- **Control remoto:** los comandos publicados desde el cliente web (`ON`, `OFF`, `AUTO` y el cambio de umbral) llegan al ESP32, que actualiza los LEDs y confirma el nuevo estado con una publicación inmediata. Así se cierra el ciclo sensor, nube, usuario y actuador.
- **Robustez:** el dispositivo ignora comandos con formato no válido sin alterar su estado. Además, el mensaje Last Will avisa a las aplicaciones cuando el dispositivo se desconecta (`offline`), lo que permite detectar fallas de comunicación.

**Limitaciones y mejoras posibles:**

- El broker público no tiene autenticación ni cifrado; en un sistema real se usaría un broker privado con usuario y contraseña y conexión TLS (puerto 8883).
- El LED azul representa al ventilador; en hardware real se usaría un relé o un transistor para manejar la carga.
- Las funciones de reconexión a WiFi y MQTT son bloqueantes: mientras el dispositivo intenta reconectarse no lee el sensor ni actualiza los LEDs. Una mejora sería reconectar de forma no bloqueante para que la alerta local siga funcionando sin conexión.
- Se podría agregar un dashboard (Node-RED, ThingSpeak o una app móvil) para graficar el histórico de mediciones y enviar notificaciones al celular.

## 5. Conclusiones

El caso práctico permitió ver cómo funciona el Internet de las Cosas en la práctica: un objeto físico (el sensor y los actuadores) se conecta a internet mediante un microcontrolador, comparte información en tiempo real y puede ser controlado a distancia. Se identificaron claramente las tres capas de la arquitectura IoT: percepción con el DHT22 y los LEDs, red con el WiFi del ESP32 y el protocolo MQTT, y aplicación con el broker y el cliente web.

El modelo publicación/suscripción de MQTT desacopla al dispositivo de las aplicaciones que consumen sus datos: el ESP32 no necesita saber quién lee la telemetría ni desde dónde se envían los comandos, lo que facilita agregar más dispositivos o usuarios. Además, combinar decisiones locales (alerta automática) con control remoto permite reaccionar de inmediato a una condición de riesgo sin depender de que un usuario esté atento, y al mismo tiempo conservar la posibilidad de intervenir a distancia.

Por último, el uso de un simulador como Wokwi permitió diseñar, programar y probar la solución sin adquirir hardware, y el mismo código puede cargarse en un ESP32 real con el Arduino IDE.

## Referencias

- Arduino. (s.f.). *Arduino Documentation*. https://docs.arduino.cc/
- Espressif Systems. (s.f.). *Arduino core for the ESP32*. https://docs.espressif.com/projects/arduino-esp32/
- Hernández Parada, M. S. (2026). *Análisis, implementación y prueba de un caso práctico con Arduino para comprender el fundamento de internet de las cosas* [Repositorio de GitHub]. GitHub. [Ver en GitHub](https://github.com/RKCbas/LENGUAJES-DE-CIENCIA-DE-DATOS-B-SICO---PRACTICAS/blob/main/Cuatrimestre%204/9%20-%20T%C3%B3picos%20de%20IA%20para%20la%20transformaci%C3%B3n%20digital/Practica%201.2/An%C3%A1lisis%2C%20implementaci%C3%B3n%20y%20prueba%20de%20un%20caso%20pr%C3%A1ctico%20con%20Arduino%20para%20comprender%20el%20fundamento%20de%20internet%20de%20las%20cosas.ipynb)
- HiveMQ. (s.f.). *MQTT Essentials*. https://www.hivemq.com/mqtt/
- O'Leary, N. (s.f.). *PubSubClient: Arduino client for MQTT*. https://pubsubclient.knolleary.net/
- Wokwi. (s.f.). *Wokwi Documentation*. https://docs.wokwi.com/